# YOLOv26 — Benchmark on OOD Dataset (22 classes)

**After training:** keep `RUN_TRAINING = False`, run **config → benchmark functions → build CSV**. That only loads `../runs/detect/yolo26n_ood*/weights/best.pt` (etc.) — **no retraining**. Then run the **display table** cell.

Compare:
- **mAP@0.5** and **mAP@0.5:0.95**
- **Precision**, **Recall**, **F1 Score**
- **Inference speed** (ms/img)
- **Model size** (MB) and **parameters** (M)
- **Per-class AP**

In [1]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.6.0+cu124
CUDA: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.6 GB


In [2]:
import pandas as pd
import numpy as np
import time
import gc
from pathlib import Path
from ultralytics import YOLO

RUN_TRAINING = False
# Exact run folders under runs/detect/ (checked first, then auto-scan if a path is missing).
BENCHMARK_RUN_FOLDERS = {
    "yolo26n": "yolo26n_ood",
    "yolo26s": "yolo26s_ood",
    "yolo26m": "yolo26m_ood",
}
BENCHMARK_VARIANTS = ["yolo26n", "yolo26s", "yolo26m"]
WORKERS = 2

DATA_YAML    = "../dataset/data.yaml"
IMG_SIZE     = 640
EPOCHS       = 100
BATCH        = 16
DEVICE       = 0
PATIENCE     = 20
RESULTS_CSV  = "benchmark_yolov26.csv"
PERCLASS_CSV = "benchmark_yolov26_perclass.csv"

CLASS_NAMES = [
    'bench', 'bicycle', 'bus', 'bus_stop', 'car', 'crutch', 'curb', 'dog',
    'fire_hydrant', 'motorcycle', 'person', 'pole', 'spherical_roadblock',
    'stairs', 'stop_sign', 'street_light', 'traffic_light', 'train', 'tree',
    'truck', 'warning_column', 'waste_container'
]

MODELS = [
    "yolo26n.pt",
    "yolo26s.pt",
    "yolo26m.pt",
]

def _safe_cuda_empty_cache():
    """Safe CUDA cache clear."""
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except:
        pass

# DEBUG: Print paths
def _runs_detect_dir() -> Path:
    """``runs/detect`` whether the notebook cwd is ``benchmark/`` or project root."""
    for rel in ("../runs/detect", "runs/detect"):
        p = Path(rel).resolve()
        if p.is_dir():
            print(f"✓ Found runs/detect at: {p}")
            return p
    print(f"✗ Cannot find runs/detect")
    return Path("../runs/detect").resolve()

print(f"Working directory: {Path.cwd()}")
print(f"DATA_YAML path: {Path(DATA_YAML).resolve()}")
_runs_detect_dir()

Working directory: c:\Users\admin\PFE\PFE-Obstacle-Detection\benchmark
DATA_YAML path: C:\Users\admin\PFE\PFE-Obstacle-Detection\data.yaml
✓ Found runs/detect at: C:\Users\admin\PFE\PFE-Obstacle-Detection\runs\detect


WindowsPath('C:/Users/admin/PFE/PFE-Obstacle-Detection/runs/detect')

In [3]:
# Debug: Check if weights files are found
from pathlib import Path

def _runs_detect_dir() -> Path:
    for rel in ("../runs/detect", "runs/detect"):
        p = Path(rel).resolve()
        if p.is_dir():
            return p
    return Path("../runs/detect").resolve()

root = _runs_detect_dir()
print(f"Root: {root}\n")

for variant in BENCHMARK_VARIANTS:
    folder = BENCHMARK_RUN_FOLDERS.get(variant)
    pinned = root / folder / "weights" / "best.pt"
    print(f"{variant}:")
    print(f"  Expected path: {pinned}")
    print(f"  Exists: {pinned.is_file()}\n")

Root: C:\Users\admin\PFE\PFE-Obstacle-Detection\runs\detect

yolo26n:
  Expected path: C:\Users\admin\PFE\PFE-Obstacle-Detection\runs\detect\yolo26n_ood\weights\best.pt
  Exists: True

yolo26s:
  Expected path: C:\Users\admin\PFE\PFE-Obstacle-Detection\runs\detect\yolo26s_ood\weights\best.pt
  Exists: True

yolo26m:
  Expected path: C:\Users\admin\PFE\PFE-Obstacle-Detection\runs\detect\yolo26m_ood\weights\best.pt
  Exists: True



In [4]:
def benchmark_model(model_name):
    """Train a model on OOD and return benchmark metrics."""
    print(f"\n{'='*60}")
    print(f"  BENCHMARK: {model_name}")
    print(f"{'='*60}")

    model = YOLO(model_name)

    model.train(
        data=DATA_YAML,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,
        name=f"{model_name.replace('.pt', '')}_ood",
        patience=PATIENCE,
        save=True,
        plots=True,
    )

    best_path = Path(model.trainer.best)
    best_model = YOLO(str(best_path))

    metrics = best_model.val(
        data=DATA_YAML, split="test", device=DEVICE, imgsz=IMG_SIZE, workers=WORKERS
    )

    test_img_dir = Path("../dataset/test/images")
    _ext = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
    test_images = [p for p in test_img_dir.glob("*.*") if p.suffix.lower() in _ext][:200]
    if not test_images:
        raise FileNotFoundError(f"No test images under {test_img_dir.resolve()}")
    latencies = []
    for _ in range(3):
        best_model(str(test_images[0]), imgsz=IMG_SIZE, device=DEVICE, verbose=False)
    for img_path in test_images:
        t0 = time.perf_counter()
        best_model(str(img_path), imgsz=IMG_SIZE, device=DEVICE, verbose=False)
        latencies.append((time.perf_counter() - t0) * 1000)

    size_mb = best_path.stat().st_size / 1e6 if best_path.exists() else 0.0
    params_m = sum(p.numel() for p in best_model.model.parameters()) / 1e6
    p = float(metrics.box.mp)
    r = float(metrics.box.mr)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0

    row = {
        "model":         model_name.replace(".pt", ""),
        "mAP@0.5":       round(float(metrics.box.map50), 4),
        "mAP@0.5:0.95":  round(float(metrics.box.map), 4),
        "precision":     round(p, 4),
        "recall":        round(r, 4),
        "F1":            round(f1, 4),
        "speed_ms/img":  round(float(np.mean(latencies)), 2),
        "size_MB":       round(size_mb, 1),
        "params_M":      round(params_m, 1),
    }

    per_class = {}
    ap50_per_class = metrics.box.ap50
    for i, name in enumerate(CLASS_NAMES):
        if i < len(ap50_per_class):
            per_class[name] = round(float(ap50_per_class[i]), 4)

    del model, best_model
    gc.collect()
    _safe_cuda_empty_cache()

    return row, per_class


def _runs_detect_dir() -> Path:
    """``runs/detect`` whether the notebook cwd is ``benchmark/`` or project root."""
    for rel in ("../runs/detect", "runs/detect"):
        p = Path(rel).resolve()
        if p.is_dir():
            return p
    return Path("../runs/detect").resolve()


def _find_newest_best_pt(variant: str):
    """Resolve weights/best.pt for ``variant`` (pinned folders first, then auto-scan)."""
    root = _runs_detect_dir()
    if not root.is_dir():
        return None
    folder = BENCHMARK_RUN_FOLDERS.get(variant)
    if folder:
        pinned = root / folder / "weights" / "best.pt"
        if pinned.is_file():
            return pinned
    cands = []
    for w in root.glob("**/weights/best.pt"):
        run_name = w.parent.parent.name
        if run_name == variant or run_name.startswith(f"{variant}_"):
            cands.append(w)
    return max(cands, key=lambda p: p.stat().st_mtime) if cands else None


def metrics_from_weights(best_pt: Path, model_key: str):
    """Test-set metrics + speed from a saved best.pt (no training)."""
    print(f"\n{'='*60}\n  METRICS ONLY: {model_key}\n  weights: {best_pt}\n{'='*60}")
    best_pt = best_pt.resolve()
    best_model = YOLO(str(best_pt))
    metrics = best_model.val(
        data=DATA_YAML, split="test", device=DEVICE, imgsz=IMG_SIZE, workers=WORKERS
    )
    test_img_dir = Path("../dataset/test/images")
    _ext = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
    test_images = [p for p in test_img_dir.glob("*.*") if p.suffix.lower() in _ext][:200]
    if not test_images:
        raise FileNotFoundError(f"No test images under {test_img_dir.resolve()}")
    latencies = []
    for _ in range(3):
        best_model(str(test_images[0]), imgsz=IMG_SIZE, device=DEVICE, verbose=False)
    for img_path in test_images:
        t0 = time.perf_counter()
        best_model(str(img_path), imgsz=IMG_SIZE, device=DEVICE, verbose=False)
        latencies.append((time.perf_counter() - t0) * 1000)
    size_mb = best_pt.stat().st_size / 1e6 if best_pt.exists() else 0.0
    params_m = sum(p.numel() for p in best_model.model.parameters()) / 1e6
    p = float(metrics.box.mp)
    r = float(metrics.box.mr)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    row = {
        "model":         model_key,
        "mAP@0.5":       round(float(metrics.box.map50), 4),
        "mAP@0.5:0.95":  round(float(metrics.box.map), 4),
        "precision":     round(p, 4),
        "recall":        round(r, 4),
        "F1":            round(f1, 4),
        "speed_ms/img":  round(float(np.mean(latencies)), 2),
        "size_MB":       round(size_mb, 1),
        "params_M":      round(params_m, 1),
    }
    per_class = {}
    ap50_per_class = metrics.box.ap50
    for i, name in enumerate(CLASS_NAMES):
        if i < len(ap50_per_class):
            per_class[name] = round(float(ap50_per_class[i]), 4)
    del best_model
    gc.collect()
    _safe_cuda_empty_cache()
    return row, per_class

In [5]:
rows = []
all_per_class = {}

if RUN_TRAINING:
    print("RUN_TRAINING=True — training each model in MODELS (slow).\n")
    for model_name in MODELS:
        try:
            row, per_class = benchmark_model(model_name)
            rows.append(row)
            all_per_class[row["model"]] = per_class
            print(f"\n  mAP@0.5       : {row['mAP@0.5']}")
            print(f"  mAP@0.5:0.95  : {row['mAP@0.5:0.95']}")
            print(f"  Precision     : {row['precision']}")
            print(f"  Recall        : {row['recall']}")
            print(f"  F1            : {row['F1']}")
            print(f"  Speed         : {row['speed_ms/img']} ms/img")
            print(f"  Size          : {row['size_MB']} MB")
            print(f"  Params        : {row['params_M']} M")
        except Exception as e:
            print(f"  SKIPPED {model_name}: {e}")
            gc.collect()
            _safe_cuda_empty_cache()
else:
    print("RUN_TRAINING=False — loading each variant's newest best.pt (no training).\n")
    print(f"Resolved runs/detect -> {_runs_detect_dir()}\n")
    for variant in BENCHMARK_VARIANTS:
        wp = _find_newest_best_pt(variant)
        if wp is None:
            print(f"  Skip {variant}: no matching best.pt under {_runs_detect_dir()} (expect folder '{variant}' or '{variant}_*')")
            continue
        try:
            row, per_class = metrics_from_weights(wp, variant)
            rows.append(row)
            all_per_class[row["model"]] = per_class
            print(f"\n  {variant}: mAP@0.5={row['mAP@0.5']}  speed={row['speed_ms/img']} ms/img")
        except Exception as e:
            print(f"  SKIPPED {variant}: {e}")
            gc.collect()
            _safe_cuda_empty_cache()

_cols = [
    "model", "mAP@0.5", "mAP@0.5:0.95", "precision", "recall", "F1",
    "speed_ms/img", "size_MB", "params_M",
]
df = pd.DataFrame(rows, columns=_cols) if rows else pd.DataFrame(columns=_cols)
df.to_csv(RESULTS_CSV, index=False)

if all_per_class:
    df_pc = pd.DataFrame(all_per_class).T
else:
    df_pc = pd.DataFrame(columns=CLASS_NAMES)
df_pc.index.name = "model"
df_pc.to_csv(PERCLASS_CSV)

print(f"\nSaved -> {RESULTS_CSV} ({len(df)} row(s))")
print(f"Saved -> {PERCLASS_CSV}")

RUN_TRAINING=False — loading each variant's newest best.pt (no training).

Resolved runs/detect -> C:\Users\admin\PFE\PFE-Obstacle-Detection\runs\detect


  METRICS ONLY: yolo26n
  weights: C:\Users\admin\PFE\PFE-Obstacle-Detection\runs\detect\yolo26n_ood\weights\best.pt
Ultralytics 8.4.30  Python-3.11.0 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO26n summary (fused): 122 layers, 2,379,126 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1246.1640.9 MB/s, size: 130.0 KB)
val: Scanning C:\Users\admin\PFE\PFE-Obstacle-Detection\dataset\test\labels.cache... 1000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1000/1000  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 8.3it/s 7.6s0.1s
                   all       1000       2945      0.664      0.572      0.611      0.436
                 bench         67        115       0.63      0.565      0.562   

In [6]:
import pandas as pd
from pathlib import Path
from IPython.display import display

RESULTS_CSV = "benchmark_yolov26.csv"

_csv = Path(RESULTS_CSV)
if not _csv.is_file() or _csv.stat().st_size < 5:
    print("No results CSV yet. Run the **build benchmark** cell above first.")
else:
    df = pd.read_csv(RESULTS_CSV)
    df.columns = [str(c).strip().lstrip("\ufeff") for c in df.columns]
    if df.empty or "model" not in df.columns:
        print("benchmark_yolov26.csv has no data rows. Run the training/benchmark cell.")
        print("Columns:", list(df.columns))
    else:
        print("=" * 60)
        print("  YOLOv26 BENCHMARK — OOD Dataset (22 classes)")
        print("=" * 60)
        print("\n", df.to_string(index=False), "\n")
        fmt = {}
        for c in df.columns:
            if c == "speed_ms/img":
                fmt[c] = "{:.1f} ms"
            elif c == "size_MB":
                fmt[c] = "{:.1f} MB"
            elif c == "params_M":
                fmt[c] = "{:.1f} M"
            elif c in ("mAP@0.5", "mAP@0.5:0.95", "precision", "recall", "F1"):
                fmt[c] = "{:.4f}"
        hi_max = [c for c in ["mAP@0.5", "mAP@0.5:0.95", "precision", "recall", "F1"] if c in df.columns]
        hi_min = [c for c in ["speed_ms/img", "size_MB", "params_M"] if c in df.columns]
        styled = df.style.set_caption("YOLOv26 Benchmark — OOD Dataset").format(fmt, na_rep="—")
        if hi_max:
            styled = styled.highlight_max(subset=hi_max, color="#2d6a2e")
        if hi_min:
            styled = styled.highlight_min(subset=hi_min, color="#1a5276")
        styled = (
            styled
            .set_properties(**{"text-align": "center", "font-size": "13px"})
            .set_table_styles([
                {"selector": "caption", "props": [("font-size", "16px"), ("font-weight", "bold"), ("padding", "10px")]},
                {"selector": "th", "props": [("background-color", "#1a1a2e"), ("color", "white"), ("padding", "8px")]},
            ])
            .hide(axis="index")
        )
        display(styled)

  YOLOv26 BENCHMARK — OOD Dataset (22 classes)

   model  mAP@0.5  mAP@0.5:0.95  precision  recall     F1  speed_ms/img  size_MB  params_M
yolo26n   0.6113        0.4361     0.6637  0.5724 0.6147         17.11      5.4       2.4
yolo26s   0.6337        0.4607     0.7147  0.5837 0.6426         19.34     20.3       9.5
yolo26m   0.6404        0.4649     0.6893  0.5998 0.6414         30.31     44.1      20.4 



model,mAP@0.5,mAP@0.5:0.95,precision,recall,F1,speed_ms/img,size_MB,params_M
yolo26n,0.6113,0.4361,0.6637,0.5724,0.6147,17.1 ms,5.4 MB,2.4 M
yolo26s,0.6337,0.4607,0.7147,0.5837,0.6426,19.3 ms,20.3 MB,9.5 M
yolo26m,0.6404,0.4649,0.6893,0.5998,0.6414,30.3 ms,44.1 MB,20.4 M
